In [ ]:
!pip install "qiskit==1.3.2"

In [12]:
import qiskit

In [13]:
qiskit.__version__

'1.3.2'

In [14]:
from qiskit_ibm_runtime import QiskitRuntimeService

In [ ]:
!pip install qiskit-ibm-runtime

In [ ]:
!pip install qiskit_aer

In [17]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import Estimator
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, Session

# Create a simple quantum circuit
qc = QuantumCircuit(2)

# h[0] = |0> + |1>/sqrt(2)
# h[1] = |0> - |1>/ sqrt(2)
qc.h(0)

# |Psi> = |00> + |11>/sqrt(2)
qc.cx(0, 1)
qc.measure_all()

# Use Qiskit Aer simulator
backend = AerSimulator()
estimator = Estimator()


print("Qiskit 1.3.2 setup is working correctly!")

In [ ]:
# Bootstrap sampling for SUM and AVG using CDKM ripple-carry adder
# Each shot = one bootstrap replication


import math, random
from typing import List, Optional, Tuple

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import CDKMRippleCarryAdder
from qiskit.transpiler import CouplingMap


# LSB of width w
def int_bits_lsb(x: int, w: int):
    return [(x >> i) & 1 for i in range(w)]  


#Set addr to i with X gates.
def set_addr(qc: QuantumCircuit, addr, i: int):
    for k, q in enumerate(addr):
        if (i >> k) & 1:
            qc.x(q)


#Uncompute
def reset_addr(qc: QuantumCircuit, addr, i: int):
    for k, q in enumerate(addr):
        if (i >> k) & 1:
            qc.x(q)


# Measure quantum register LSB-first into a classical register
def measure_lsb(qc: QuantumCircuit, qreg, creg: ClassicalRegister, creg_offset: int = 0):
    for i, q in enumerate(qreg):
        qc.measure(q, creg[creg_offset + i])


# Reversible QROM
def qrom_load_low_bits(qc: QuantumCircuit, addr, val, table: List[int], n_write_bits: int):
    nA = len(addr)
    padded = list(table) + [0] * ((1 << nA) - len(table))
    for a, y in enumerate(padded):
        flips = []
        for k in range(nA):
            if ((a >> k) & 1) == 0:   #Shifts a to the right by k bits
                qc.x(addr[k]) 
                flips.append(k)
        for j in range(n_write_bits):
            if (y >> j) & 1:
                qc.mcx(list(addr), val[j])
        for k in flips:
            qc.x(addr[k])


# CDKM ripple-carry adder

#Implements b := a + b (mod 2^w)
def add_with_cdkm(qc: QuantumCircuit, a_reg, b_reg, helper):
    w = len(a_reg)
    adder = CDKMRippleCarryAdder(w, kind='fixed')
    qc.append(adder, qargs=[*a_reg, *b_reg, helper])


# Deterministic Classical Resampling (classical indices) – kept for validation

def sum_indices(values: List[int], idxs: List[int]) -> Tuple[QuantumCircuit, int]:

    n = len(values)
    draws = len(idxs)
    a_bits    = math.ceil(math.log2(max(1, n)))
    data_bits = max(1, max(values).bit_length())
    w         = data_bits + math.ceil(math.log2(max(1, draws)))   # accumulator width

    addr  = QuantumRegister(a_bits, 'addr')
    val_w = QuantumRegister(w, 'val')    
    acc   = QuantumRegister(w, 'acc')
    help_anc = QuantumRegister(1, 'anc')
    cacc  = ClassicalRegister(w, 'c_acc')

    qc = QuantumCircuit(addr, val_w, acc, help_anc, cacc, name ='SUM_bootstrap')

    for i in idxs:
        set_addr(qc, addr, i)
        qrom_load_low_bits(qc, addr, val_w, values, n_write_bits = data_bits)  # val_w <-- y_i (low bits)
        add_with_cdkm(qc, list(val_w), list(acc), help_anc[0])          # acc += val_w
        qrom_load_low_bits(qc, addr, val_w, values, n_write_bits = data_bits)  # unload
        reset_addr(qc, addr, i)

    measure_lsb(qc, acc, cacc, 0)  # int(bitstring,2) == sum
    return qc, w


# Quantum resampling 

def sum_quantum_resampling(values: List[int], draws: int) -> Tuple[QuantumCircuit, int, int]:
    
    n = len(values)
    a_bits    = math.ceil(math.log2(max(1, n)))
    data_bits = max(1, max(values).bit_length())
    w         = data_bits + math.ceil(math.log2(max(1, draws)))  

    addr_regs = [QuantumRegister(a_bits, f'addr{j}') for j in range(draws)]
    val_w = QuantumRegister(w, 'val')
    acc   = QuantumRegister(w, 'acc')
    help_anc = QuantumRegister(1, 'anc')

    total_cbits = w + draws * a_bits
    c_all  = ClassicalRegister(total_cbits, 'c_all')

    qc = QuantumCircuit(*addr_regs, val_w, acc, help_anc, c_all, name = 'SUM_qresample')    #*addr_regs takes in tuple values

    # Prepare uniform superposition for each draw
    for j in range(draws):
        for k in range(a_bits):
            qc.h(addr_regs[j][k])

    # For each draw, QROM load --> CDKM add --> QROM unload
    for j in range(draws):
        qrom_load_low_bits(qc, addr_regs[j], val_w, values, n_write_bits = data_bits)
        add_with_cdkm(qc, list(val_w), list(acc), help_anc[0])
        qrom_load_low_bits(qc, addr_regs[j], val_w, values, n_write_bits = data_bits)

    # measurement
    offset = 0
    measure_lsb(qc, acc, c_all, offset)
    offset += w
    for j in range(draws):
        measure_lsb(qc, addr_regs[j], c_all, offset); offset += a_bits
    return qc, w, a_bits



_SIM = AerSimulator(method="matrix_product_state")

def run_and_int(qc: QuantumCircuit, shots: int = 1):
    
    tqc = transpile(
        qc, _SIM,
        coupling_map = CouplingMap.from_full(qc.num_qubits),
        layout_method = "trivial",
        routing_method = "none",
        optimization_level = 0,
    )
    result = _SIM.run(tqc, shots = shots).result()
    counts = result.get_counts()
    values = []
    for bitstr, cnt in counts.items():
        values.extend([int(bitstr.replace(' ', ''), 2)] * cnt)  # remove spaces if there's any
    return counts, values  


# Deterministic bootstrap sampling (for validation)

def one_replicate(values: List[int], idxs: List[int], where_pred: Optional[List[int]] = None):
    chosen_vals = [values[i] for i in idxs]
    if where_pred is None:
        classical_sum = sum(chosen_vals)
        qc, _ = sum_indices(values, idxs)
        _, vals = run_and_int(qc, shots = 1)
        quantum_sum = vals[0]
        denom = len(idxs)
    else:
        classical_sum = sum(values[i] for i in idxs if where_pred[i] == 1)
        qc, _ = sum_indices_where(values, idxs, where_pred)
        _, vals = run_and_int(qc, shots = 1)
        quantum_sum = vals[0]
        denom = sum(where_pred[i] for i in idxs)

    print(f"Indices: {idxs}")
    print(f"Values : {chosen_vals}")
    print(f"Classical SUM = {classical_sum} and Quantum SUM = {quantum_sum}")
    avg_c = classical_sum / denom if denom > 0 else 0.0
    avg_q = quantum_sum  / denom if denom > 0 else 0.0
    print(f"AVG: classical = {avg_c:.6f}  quantum = {avg_q:.6f} (denominator = {denom})")
    return classical_sum, quantum_sum


def bootstrap(values: List[int], draws: int, B: int, seed: int, where_pred: Optional[List[int]] = None):
    rng = random.Random(seed)
    for b in range(1, B + 1):
        idxs = [rng.randrange(len(values)) for _ in range(draws)]
        print(f"\n[Resample {b}]")
        one_replicate(values, idxs, where_pred)



# Decoders (classical readout --> tuples)

def decode_readings(readings: List[int], w_sum: int, a_bits: int, draws: int):
    """Decode integer into (sum, [idx0..idx_{draws-1}]) per shot"""
    mask_sum  = (1 << w_sum) - 1
    mask_addr = (1 << a_bits) - 1
    out = []
    for v in readings:
        s = v & mask_sum
        idxs = []
        x = v >> w_sum
        for _ in range(draws):
            idxs.append(x & mask_addr)
            x >>= a_bits
        out.append((s, idxs))
    return out


# Quantum bootstrap

def bootstrap_quantum(values: List[int], draws: int, B: int):
    """
    Quantum resampling. Returns list of (SUM, [idxs]).
    AVG is sum / draws.
    """
    qc, w_sum, a_bits = sum_quantum_resampling(values, draws)
    _, readings = run_and_int(qc, shots = B)
    return decode_readings(readings, w_sum, a_bits, draws)



#Draw circuit
from qiskit.visualization import circuit_drawer


def _prepare_for_draw(circ, *, expand_composites: bool, basis_level: bool, hide_meas: bool):
    c = circ
    if hide_meas:
        c = c.remove_final_measurements(inplace = False)

    if expand_composites:
        c = c.decompose(gates_to_decompose = ['cdkm_ripple_carry_adder'], reps = 1)

    if basis_level:
        
        c = transpile(
            c,
            basis_gates = ['x', 'cx', 'ccx'],
            optimization_level = 0,
            routing_method = 'none',
            layout_method = 'trivial',
        )
    return c


def draw_cdkm(width: int, *, expand = True, basis_level = False,
                   fold = -1, scale = 0.9, vertical_compression = 'medium', save_path = None):
    a = QuantumRegister(width, 'a')
    b = QuantumRegister(width, 'b')
    anc = QuantumRegister(1, 'anc')
    qc = QuantumCircuit(a, b, anc, name = f'CDKM(w = {width})')
    add_with_cdkm(qc, list(a), list(b), anc[0])

    circ = _prepare_for_draw(qc, expand_composites = expand,
                             basis_level = basis_level, hide_meas = True)

    
    try:
        print("Operations:", circ.count_ops())
    except Exception:
        pass

    fig = circuit_drawer(
        circ, output ='mpl', scale = scale, fold = fold,
        vertical_compression = vertical_compression,
        idle_wires = False, plot_barriers = False, reverse_bits = False,
    )
    if save_path:
        fig.savefig(save_path, dpi = 300, bbox_inches = 'tight', facecolor = 'white')
    return fig



def draw_full_pipeline(values, draws, *,
                                     draws_render = None,
                                     expand_composites = False,
                                     basis_level = False,
                                     hide_meas = False,
                                     fold = 160,
                                     scale = 0.08,
                                     vertical_compression = 'high',
                                     save_path = None):
    """
    draws_render: if set, render a smaller fig (e.g., 2 or 3) to keep figures compact.
    scale: 0.05–0.12 is a good range for large circuits.
    fold: wrap rows to balance width/height.
    """
    d = draws_render if draws_render is not None else draws
    qc, _, _ = sum_quantum_resampling(values, d)
    circ = _prepare_for_draw(qc, expand_composites = expand_composites,
                             basis_level = basis_level, hide_meas = hide_meas)

    fig = circuit_drawer(
        circ,
        output ='mpl',
        scale = scale,
        fold = fold,
        vertical_compression = vertical_compression,
        cregbundle = True,
        idle_wires = False,
        plot_barriers = False,
        reverse_bits = False,
    )
    if save_path:
        fig.savefig(save_path, dpi = 300, bbox_inches = 'tight')
    return fig




if __name__ == "__main__":

    base  = [3, 9, 9, 10, 6, 20, 10, 6]
    n     = len(base)
    draws = n
    B     = 8

    # print("\n Deterministic bootstrap (classical resampling)")
    bootstrap(base, draws, B, seed = 42)

    # print("\n Quantum resampling with indices")
    reps = bootstrap_quantum(base, draws, B)  
    for i, (s, idxs) in enumerate(reps, 1):
        avg = s / draws
        print(f"[Shot {i}] SUM = {s:2d}  AVG = {avg:.3f}  IDX = {idxs}")
        
    
    draw_full_pipeline(
        base, 
        draws = 8,
        draws_render = 1,               # only 1 draw
        expand_composites = False,      
        fold = -1,                      
        scale = 0.20,                   
        save_path = 'pipeline_1draw.svg'
    )

    w = max(1, max(base).bit_length()) + math.ceil(math.log2(draws))
    fig = draw_cdkm(w, expand = True, basis_level = True, fold = -1, scale = 0.9)


    from pathlib import Path
    Path("figures").mkdir(exist_ok = True)

    fig.savefig("figures/cdkm_w8.png", dpi = 300, bbox_inches = 'tight', pad_inches = 0.01, facecolor = 'white')





 Deterministic bootstrap (classical resampling)

[Resample 1]
Indices: [1, 0, 4, 3, 3, 2, 1, 1]
Values : [9, 3, 6, 10, 10, 9, 9, 9]
Classical SUM = 65 and Quantum SUM = 65
AVG: classical = 8.125000  quantum = 8.125000 (denominator = 8)

[Resample 2]
Indices: [6, 0, 0, 1, 3, 3, 0, 3]
Values : [10, 3, 3, 9, 10, 10, 3, 10]
Classical SUM = 58 and Quantum SUM = 58
AVG: classical = 7.250000  quantum = 7.250000 (denominator = 8)

[Resample 3]
Indices: [6, 3, 7, 4, 0, 2, 6, 5]
Values : [10, 10, 6, 6, 3, 9, 10, 20]
Classical SUM = 74 and Quantum SUM = 74
AVG: classical = 9.250000  quantum = 9.250000 (denominator = 8)

[Resample 4]
Indices: [4, 2, 3, 5, 1, 1, 6, 1]
Values : [6, 9, 10, 20, 9, 9, 10, 9]
Classical SUM = 82 and Quantum SUM = 82
AVG: classical = 10.250000  quantum = 10.250000 (denominator = 8)

[Resample 5]
Indices: [5, 5, 4, 0, 7, 1, 6, 1]
Values : [20, 20, 6, 3, 6, 9, 10, 9]
Classical SUM = 83 and Quantum SUM = 83
AVG: classical = 10.375000  quantum = 10.375000 (denominator = 8)

